In [1]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from collections import Counter
import random

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_35402/511043635.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [2]:
def jaccard_similarity(labels1, labels2):
    """Вычисляет Jaccard similarity между двумя бинарными векторами"""
    intersection = np.sum(np.array(labels1) * np.array(labels2))
    union = np.sum((np.array(labels1) + np.array(labels2)) > 0)
    return intersection / union if union > 0 else 0

def oversample_rare_classes_dataset(train_dataset, target_count_per_class=200, num_classes=8):
    """Увеличивает количество примеров для редких классов в datasets.Dataset"""
    # Извлекаем метки
    all_labels = train_dataset['label']
    
    # Считаем текущее распределение
    class_counts = Counter()
    for labels in all_labels:
        # Если метки в виде списка/массива
        for class_idx, has_label in enumerate(labels):
            if has_label == 1:
                class_counts[class_idx] += 1
    
    # Создаем списки для аугментированных данных
    augmented_texts = list(train_dataset['text'])
    augmented_labels = list(train_dataset['label'])
    
    for class_idx in range(num_classes):
        current_count = class_counts.get(class_idx, 0)
        if current_count < target_count_per_class:
            # Находим индексы примеров с этим классом
            indices_with_class = [
                i for i, labels in enumerate(train_dataset['label']) 
                if labels[class_idx] == 1
            ]
            
            if indices_with_class:
                need_to_add = min(
                    target_count_per_class - current_count,
                    len(indices_with_class) * 3  # Ограничиваем
                )
                
                for _ in range(need_to_add):
                    idx = random.choice(indices_with_class)
                    original_text = train_dataset['text'][idx]
                    original_labels = train_dataset['label'][idx].copy()
                    
                    # Простая аугментация текста
                    augmented_text = original_text + f" (вариант_{random.randint(1,100)})"
                    
                    augmented_texts.append(augmented_text)
                    augmented_labels.append(original_labels)
    
    # Создаем новый Dataset
    balanced_dataset = Dataset.from_dict({
        "text": augmented_texts,
        "label": augmented_labels
    })
    
    print(f"Исходный размер: {len(train_dataset)}, После балансировки: {len(balanced_dataset)}")
    return balanced_dataset

def create_triplets_from_dataset(train_dataset, pos_threshold=0.5, neg_threshold=0.3, max_triplets=5000):
    """Создает триплеты из datasets.Dataset"""
    triplets = []
    data_list = list(zip(train_dataset['text'], train_dataset['label']))
    
    # Для больших датасетов берем случайную подвыборку
    if len(data_list) > 1000:
        data_list = random.sample(data_list, 1000)
    
    for i, (text_anchor, labels_anchor) in enumerate(data_list):
        pos_candidates = []
        neg_candidates = []
        
        for j, (text_candidate, labels_candidate) in enumerate(data_list):
            if i == j:
                continue
            
            sim = jaccard_similarity(labels_anchor, labels_candidate)
            
            if sim >= pos_threshold:
                pos_candidates.append(text_candidate)
            elif sim <= neg_threshold:
                neg_candidates.append(text_candidate)
        
        # Создаем триплеты
        for positive in pos_candidates[:3]:  # Ограничиваем для скорости
            for negative in neg_candidates[:3]:
                triplets.append(InputExample(texts=[text_anchor, positive, negative]))
                
                # Ограничиваем общее количество триплетов
                if len(triplets) >= max_triplets:
                    print(f"Достигнут лимит триплетов: {max_triplets}")
                    return triplets
    
    print(f"Создано {len(triplets)} триплетов")
    return triplets

def find_optimal_thresholds(classifier, val_embeddings, val_labels, num_classes=8):
    """Подбирает оптимальный порог для каждого класса"""
    thresholds = []
    
    for class_idx in range(num_classes):
        # Получаем вероятности для класса
        probas = classifier.predict_proba(val_embeddings)[class_idx][:, 1]
        true_labels = [labels[class_idx] for labels in val_labels]
        
        # Подбираем порог
        best_threshold = 0.5
        best_f1 = 0
        
        for threshold in np.arange(0.1, 0.95, 0.05):
            preds = (probas > threshold).astype(int)
            f1 = f1_score(true_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
        
        thresholds.append(best_threshold)
        print(f"Class {class_idx}: threshold = {best_threshold:.2f}, F1 = {best_f1:.3f}")
    
    return thresholds

def predict_with_thresholds(model, classifier, thresholds, texts):
    """Предсказывает классы с использованием оптимальных порогов"""
    embeddings = model.encode(texts)
    probabilities = classifier.predict_proba(embeddings)
    
    predictions = []
    for i in range(len(texts)):
        pred = []
        for class_idx in range(len(thresholds)):
            prob = probabilities[class_idx][i, 1]
            pred.append(1 if prob > thresholds[class_idx] else 0)
        predictions.append(pred)
    
    return predictions

In [3]:
import pandas as pd

In [4]:
df=pd.read_csv("/home/sasha/Python/VKR/pytorch_bert/new_ds.csv")

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['text'].to_list(),df.drop(columns=['text']).values)

In [6]:
from datasets import Dataset
train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": y_train  
})
eval_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test
})

In [7]:
balanced_train_dataset = oversample_rare_classes_dataset(
    train_dataset, 
    target_count_per_class=200,  # Настройте под ваши данные
    num_classes=8
)

Исходный размер: 1711, После балансировки: 2013


In [8]:
train_triplets = create_triplets_from_dataset(
    balanced_train_dataset,
    pos_threshold=0.5,
    neg_threshold=0.3,
    max_triplets=5000  # 
)


Достигнут лимит триплетов: 5000


In [9]:
from sentence_transformers import models, SentenceTransformer

word_embedding_model = models.Transformer("ai-forever/FRIDA")

# ВКЛЮЧАЕМ ГРАДИЕНТНЫЙ ЧЕКПОИНТИНГ
word_embedding_model.auto_model.gradient_checkpointing_enable()

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

/tmp/ipykernel_35402/30100580.py:1: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import models, SentenceTransformer
Loading weights: 100%|██████████| 219/219 [00:00<00:00, 16940.59it/s]
/tmp/ipykernel_35402/30100580.py:8: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())


In [10]:
train_dataloader = DataLoader(train_triplets, shuffle=True, batch_size=2)
train_loss = losses.TripletLoss(model, triplet_margin=1)
model.max_seq_length = 128
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=20,
    use_amp=True,
    warmup_steps=100,
    output_path="./frida_multilabel_finetuned",
    show_progress_bar=True
)

Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 11.63 GiB of which 90.50 MiB is free. Including non-PyTorch memory, this process has 10.54 GiB memory in use. Of the allocated memory 9.89 GiB is allocated by PyTorch, and 519.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)